# Bank Conversion Prediction
* **Business Problem:** Our customer, a Bank Institution, wants to optimize sales and their manager's time when selling financial products. The product for this project is a Term Direct Deposit.
  * A **fixed-term** investment where money is deposited into an account at a financial institution. Term deposits are also known as certificates of deposit (CDs) or time deposits. Term deposits typically have higher interest rates than traditional savings accounts, but the funds are not accessible until the term ends.

* **Proposed Solution:** This project aims to create classifier to predict the probability of a customer to convert when offered a financial product (direct term deposit) via a phone call.


* The Dataset can be found in [this link, from UCI DS Repository](https://archive.ics.uci.edu/dataset/222/bank+marketing)

* This project is an example of the use of the framework CRISP-DM in practice.

In [2]:
#Data Wrangling
import pandas as pd
import numpy as np

# Data Viz
import matplotlib.pyplot as plt
import seaborn as sns
from ucimlrepo import fetch_ucirepo
sns.set()

# Transformations and Modeling
from sklearn.model_selection import train_test_split



## Dataset
Moro, S., Rita, P., & Cortez, P. (2014). Bank Marketing [Dataset]. UCI Machine Learning Repository. https://doi.org/10.24432/C5K306.

In [3]:
# fetch dataset 
bank_marketing = fetch_ucirepo(id=222) 

# data (as pandas dataframes)
df = pd.concat([bank_marketing.data.features, bank_marketing.data.targets],axis=1)

#Snapshot of first 5 records
df.sample(5)

,age,job,marital,education,default,balance,housing,loan,contact,day_of_week,month,duration,campaign,pdays,previous,poutcome,y
15022,27,admin.,single,secondary,no,180,no,yes,cellular,17,jul,114,1,-1,0,NaN,no
15611,29,technician,single,tertiary,no,-229,yes,no,cellular,21,jul,416,4,-1,0,NaN,no
10879,40,admin.,married,secondary,no,35,no,no,NaN,17,jun,78,1,-1,0,NaN,no
32892,49,entrepreneur,married,secondary,no,687,yes,no,cellular,17,apr,772,1,-1,0,NaN,yes
12750,46,NaN,married,NaN,no,2911,no,no,telephone,7,jul,41,9,-1,0,NaN,no


In [4]:
df.shape

(45211, 17)

In [9]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 45211 entries, 0 to 45210
Data columns (total 17 columns):
 #   Column       Non-Null Count  Dtype 
---  ------       --------------  ----- 
 0   age          45211 non-null  int64 
 1   job          44923 non-null  object
 2   marital      45211 non-null  object
 3   education    43354 non-null  object
 4   default      45211 non-null  object
 5   balance      45211 non-null  int64 
 6   housing      45211 non-null  object
 7   loan         45211 non-null  object
 8   contact      32191 non-null  object
 9   day_of_week  45211 non-null  int64 
 10  month        45211 non-null  object
 11  duration     45211 non-null  int64 
 12  campaign     45211 non-null  int64 
 13  pdays        45211 non-null  int64 
 14  previous     45211 non-null  int64 
 15  poutcome     8252 non-null   object
 16  y            45211 non-null  object
dtypes: int64(7), object(10)
memory usage: 5.9+ MB


### Keys and duplicates
The UCI Bank Marketing dataset does not have a unique identifier column since the goal is not to identify the customer in real world, but to predict with the features provided during the campaign to predict if the customer would subscribe to the offer. For our scenario, a duplicated record does not necessarily mean the same customer appears twice. Two different customers can have identical values across all recorded features. Also, repeated campaign contacts may be legitimate records, so investigation of the duplicates is necessary before any deletion.

In [5]:
print("full-row duplicates:", df.duplicated().sum())

full-row duplicates: 0


In [6]:
# Split in train and test sets
X_train, X_test, y_train, y_test = train_test_split(df.drop('y', axis=1),
                                                    df['y'],
                                                    test_size=0.2,
                                                    stratify=df['y'],
                                                    random_state=42)

# train
df_train = pd.concat([X_train, y_train], axis=1)

# test
df_test = pd.concat([X_test, y_test], axis=1)


## [CRISP-DM] Data Understanding

Understanding the data is crucial in getting insights of what drives an output can help us during the modeling phase.

### Statistical Description

Let's start our exploration with the statistical description of our data

In [12]:
df.describe(include="all").T.round(2)

,count,unique,top,freq,mean,std,min,25%,50%,75%,max
age,45211.0,NaN,NaN,NaN,40.93621,10.618762,18.0,33.0,39.0,48.0,95.0
job,44923,11,blue-collar,9732,NaN,NaN,NaN,NaN,NaN,NaN,NaN
marital,45211,3,married,27214,NaN,NaN,NaN,NaN,NaN,NaN,NaN
education,43354,3,secondary,23202,NaN,NaN,NaN,NaN,NaN,NaN,NaN
default,45211,2,no,44396,NaN,NaN,NaN,NaN,NaN,NaN,NaN
balance,45211.0,NaN,NaN,NaN,1362.272058,3044.765829,-8019.0,72.0,448.0,1428.0,102127.0
housing,45211,2,yes,25130,NaN,NaN,NaN,NaN,NaN,NaN,NaN
loan,45211,2,no,37967,NaN,NaN,NaN,NaN,NaN,NaN,NaN
contact,32191,2,cellular,29285,NaN,NaN,NaN,NaN,NaN,NaN,NaN
day_of_week,45211.0,NaN,NaN,NaN,15.806419,8.322476,1.0,8.0,16.0,21.0,31.0


**Lets calculate the conversion rate for our dataset**

In [9]:
print(df['y'].value_counts())
print(f"Conversion rate: {round(((df['y'] == 'yes').mean())*100,2)} %")

print(df['default'].value_counts())
print(f"Payment default: {round(((df['default'] == 'yes').mean())*100,2)} %")

y
no     39922
yes     5289
Name: count, dtype: int64
Conversion rate: 11.7 %
default
no     44396
yes      815
Name: count, dtype: int64
Payment default: 1.8 %


In [7]:
#Statistical description of training data

df_train.describe(include='all').T.round(2)

,count,unique,top,freq,mean,std,min,25%,50%,75%,max
age,36168.0,NaN,NaN,NaN,40.892999,10.627075,18.0,33.0,39.0,48.0,95.0
job,35934,11,blue-collar,7830,NaN,NaN,NaN,NaN,NaN,NaN,NaN
marital,36168,3,married,21771,NaN,NaN,NaN,NaN,NaN,NaN,NaN
education,34686,3,secondary,18561,NaN,NaN,NaN,NaN,NaN,NaN,NaN
default,36168,2,no,35521,NaN,NaN,NaN,NaN,NaN,NaN,NaN
balance,36168.0,NaN,NaN,NaN,1365.49342,3068.5435,-8019.0,74.0,451.0,1430.25,102127.0
housing,36168,2,yes,20182,NaN,NaN,NaN,NaN,NaN,NaN,NaN
loan,36168,2,no,30355,NaN,NaN,NaN,NaN,NaN,NaN,NaN
contact,25782,2,cellular,23465,NaN,NaN,NaN,NaN,NaN,NaN,NaN
day_of_week,36168.0,NaN,NaN,NaN,15.817961,8.33198,1.0,8.0,16.0,21.0,31.0


In [10]:
#Lets calculate the conversion rate for training dataset

print(df_train['y'].value_counts())
print(f"Conversion rate: {round(((df_train['y'] == 'yes').mean())*100,2)} %")

print(df_train['default'].value_counts())
print(f"Payment default: {round(((df_train['default'] == 'yes').mean())*100,2)} %")


y
no     31937
yes     4231
Name: count, dtype: int64
Conversion rate: 11.7 %
default
no     35521
yes      647
Name: count, dtype: int64
Payment default: 1.79 %


**Highlights:**
* Age of the customers is 40 years old on average. Distribution skewed to the right.
* More than 20% of the customers are blue-collar workers.
* Most of the customers are married, with secondary level education, own a house loan.
* Only ~2% had payment default.
* Conversion Rate ~ 11.7%

Now let's look at the distribution of our target variable ( *y* )


**Analysis** | The target variable is very unbalanced since we have a very less conversion rate of 11.7%